# New Version: congestion-sensitive feature analysis

این نوت‌بوک همهٔ ستون‌های اصلی را نگه می‌دارد و حساسیت ویژگی‌ها را **درون هر برچسب** مقایسه می‌کند.
هیچ ویژگی‌ای بر اساس ضعیف‌کردن بنچمارک انتخاب نمی‌شود. رتبه‌بندی توصیفی است و هنوز انتخاب نهایی ویژگی یا ارزیابی مدل نیست.

Source revision: `425fcc486c797e0eb0e542d6c5f176b22d4d50ca`.
Files: `CDR_MLC/DATASETS/CDR-MLC/New_Version/*.flow` (CSV text exported by Argus).

Run all cells from the repository root or `CDR_MLC`. Dependencies: Python, pandas, numpy, scipy, matplotlib, seaborn, Jupyter.
If needed, install in your environment: `python -m pip install pandas numpy scipy matplotlib seaborn jupyter`.
Reports are written under `CDR_MLC/analysis_outputs/new_version_features`; original data is never overwritten.


## Verified initial run

All 9 code cells were executed sequentially on the 15 downloaded source files (120,568 records); Git blob hashes matched the pinned revision. Execution used Python with a non-interactive plotting backend.

Development-only macro KS ranking: TcpRtt 0.4824, AckDat 0.4768, SynAck 0.4301. These are observed distribution differences, not causal estimates. Sixteen measurement fields plus Label are entirely empty. Dur and Proto headers are repeated. IdleTime is constant within each file with values around 1.79 billion.

Outputs are cleared in Git to keep the notebook small. Run All regenerates tables, plots and CSV reports locally.


In [ ]:
from pathlib import Path
from collections import Counter
from itertools import combinations
import csv, hashlib, json, re, platform
import numpy as np
import pandas as pd
import scipy
from scipy.stats import ks_2samp, mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

LEVELS = ['Low', 'Medium', 'High']
CLASSES = ['HTTP', 'SFTP', 'SMTP', 'SSH', 'Video']
DEV_FRACTION = 0.70
MIN_VALID = 30
MIN_COVERAGE = 0.80
SEED = 42
SOURCE_COMMIT = '425fcc486c797e0eb0e542d6c5f176b22d4d50ca'
DATA_DIR_OVERRIDE = None  # optional absolute Path to New_Version
bases = [Path.cwd(), *Path.cwd().parents]
candidates = [b / rel for b in bases for rel in (
    'CDR_MLC/DATASETS/CDR-MLC/New_Version', 'DATASETS/CDR-MLC/New_Version')]
DATA_DIR = Path(DATA_DIR_OVERRIDE) if DATA_DIR_OVERRIDE else next((p for p in candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Set DATA_DIR_OVERRIDE to the New_Version directory.')
OUT = DATA_DIR.parents[2] / 'analysis_outputs' / 'new_version_features'
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)
sns.set_theme(style='whitegrid')
print('Data:', DATA_DIR, '\nReports:', OUT)
print('Versions:', platform.python_version(), pd.__version__, np.__version__, scipy.__version__)


## 1. Read all files without silently dropping malformed records
Duplicate header names are preserved as `.1`, `.2`, etc. File labels are provenance metadata; they are not model inputs.
A file is treated as a sequence identifier, not proof of an independent experimental replicate.


In [ ]:
frames, inventory, header_audit = [], [], []
for path in sorted(DATA_DIR.glob('*.flow')):
    match = re.fullmatch(r'(HTTP|SFTP|SMTP|SSH|Video)_(Low|Medium|High)', path.stem)
    if not match:
        raise ValueError(f'Unexpected filename: {path.name}')
    traffic, level = match.groups()
    with path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.reader(stream)
        header = next(reader)
        bad = [(i, len(row)) for i, row in enumerate(reader, 2) if row and len(row) != len(header)]
    if bad:
        raise ValueError(f'Malformed rows in {path.name}: {bad[:10]}')
    df = pd.read_csv(path, encoding='utf-8-sig', low_memory=False, on_bad_lines='error')
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include='object'):
        df[col] = df[col].str.strip().replace('', np.nan)
    duplicate_rows = int(df.duplicated().sum())
    for name, count in Counter(header).items():
        if count > 1:
            identical = all(df[name].fillna('<NA>').equals(df[f'{name}.{i}'].fillna('<NA>')) for i in range(1,count))
            header_audit.append(dict(file=path.name, field=name, occurrences=count, copies_identical=identical))
    df['traffic_label'], df['congestion_level'] = traffic, level
    df['run_id'], df['source_file'] = path.stem, path.name
    df['source_row'] = np.arange(2, len(df)+2)
    df['timestamp'] = pd.to_datetime(df['StartTime'], format='%Y/%m/%d %H:%M:%S.%f', errors='coerce')
    if df.timestamp.isna().any():
        raise ValueError(f'Unparsed timestamps in {path.name}')
    inventory.append(dict(file=path.name, traffic_label=traffic, congestion_level=level,
        records=len(df), original_columns=len(header), duplicate_rows=duplicate_rows,
        bytes=path.stat().st_size, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
        first_time=df.timestamp.min(), last_time=df.timestamp.max(),
        start_records=int(df.Cause.eq('Start').sum()), status_records=int(df.Cause.eq('Status').sum())))
    frames.append(df)
if len(frames) != 15:
    raise ValueError(f'Expected 15 files; found {len(frames)}')
data = pd.concat(frames, ignore_index=True)
assert set(zip(data.traffic_label, data.congestion_level)) == set((c,l) for c in CLASSES for l in LEVELS)
original = [c for c in frames[0].columns if c not in ['traffic_label','congestion_level','run_id','source_file','source_row','timestamp']]
inventory = pd.DataFrame(inventory)
header_audit = pd.DataFrame(header_audit)
display(inventory.pivot(index='traffic_label', columns='congestion_level', values='records').reindex(columns=LEVELS))
print('Total Argus records:', len(data))
display(header_audit)
inventory.to_csv(OUT/'inventory.csv', index=False)
header_audit.to_csv(OUT/'duplicate_headers.csv', index=False)


## 2. Quality and semantics audit
Missing values are **not** replaced with zero. Zero handshake measurements are reported separately; they are not automatically assumed to be valid measured zero latency.
`IdleTime` is retained but excluded from the primary ranking until its unusually large values are explained.
`Start`/`Status` are export records, not necessarily independent TCP connections. Do not treat row counts as session counts.


In [ ]:
quality = []
for (run, group) in data.groupby('run_id', sort=False):
    for col in original:
        s = group[col]
        n = pd.to_numeric(s, errors='coerce')
        quality.append(dict(run_id=run, feature=col, missing_fraction=s.isna().mean(),
            unique_values=s.nunique(dropna=True), numeric_valid_fraction=n.notna().mean(),
            invalid_numeric_nonmissing=int((s.notna() & n.isna()).sum()),
            zero_fraction=n.eq(0).mean(), min=n.min(), max=n.max()))
quality = pd.DataFrame(quality)
empty = [c for c in original if data[c].isna().all()]
constant = [c for c in original if data[c].nunique(dropna=True)==1]
print('Entirely empty:', empty)
print('Globally constant:', constant)
display(data.groupby(['traffic_label','congestion_level','Cause']).size().unstack(fill_value=0))
display(quality[quality.feature.isin(['TcpRtt','SynAck','AckDat','IdleTime'])])
quality.to_csv(OUT/'feature_quality.csv', index=False)
# Check the measured identity rather than counting all three as independent information.
residual = pd.to_numeric(data.TcpRtt)-pd.to_numeric(data.SynAck)-pd.to_numeric(data.AckDat)
print('TcpRtt ≈ SynAck + AckDat, absolute tolerance 2e-6:', np.isclose(residual,0,atol=2e-6).mean())


## 3. Development/holdout separation before feature ranking
Within each file, use the first 70% of chronological records for development and reserve the last 30%.
Purge any 5-tuple appearing on both sides. This is conservative: port reuse may purge unrelated sessions too.
A 5-tuple is **not** used as a unique session identifier. Whole repeated runs should replace this split when available.
The holdout values are not used to rank features. General quality checks above cover all data.


In [ ]:
tuple_cols = ['SrcAddr','DstAddr','Proto','Sport','Dport']
parts=[]
for run,g in data.groupby('run_id', sort=False):
    g=g.sort_values(['timestamp','source_row'], kind='stable').copy()
    # Canonical bidirectional endpoint key, so reverse records cannot cross the split.
    left=g.SrcAddr.astype(str)+':'+g.Sport.astype(str)
    right=g.DstAddr.astype(str)+':'+g.Dport.astype(str)
    g['tuple_key']=np.where(left<=right,left+'|'+right,right+'|'+left)+'|'+g.Proto.astype(str)
    cut=int(len(g)*DEV_FRACTION)
    g['partition']=np.where(np.arange(len(g))<cut,'development','holdout')
    crossing=set(g.iloc[:cut].tuple_key)&set(g.iloc[cut:].tuple_key)
    g.loc[g.tuple_key.isin(crossing),'partition']='purged'
    assert not (set(g.loc[g.partition.eq('development'),'tuple_key']) & set(g.loc[g.partition.eq('holdout'),'tuple_key']))
    parts.append(g)
data=pd.concat(parts,ignore_index=True)
dev=data[data.partition.eq('development')].copy()
split_counts=data.groupby(['run_id','partition']).size().unstack(fill_value=0)
display(split_counts)
split_counts.to_csv(OUT/'split_counts.csv')
data[['source_file','source_row','run_id','partition']].to_csv(OUT/'split_manifest.csv',index=False)


## 4. Candidate roles — retain everything, distinguish interpretation
Primary ranking includes all available numeric behavioral fields, including size and volume; none are removed to disadvantage a benchmark.
Addresses, ports, absolute time, labels, export categories and TTL are metadata/context, not primary congestion candidates.
Constant/empty/duplicate columns are reported rather than silently interpreted as informative.


In [ ]:
context={'StartTime','SrcAddr','DstAddr','Sport','Dport','Proto','Proto.1','Label','Flgs','Cause','Dir','State','TcpOpt','sTtl','dTtl'}
roles=[]
for c in original:
    numeric=pd.to_numeric(dev[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    if c in context: role='metadata_or_context'
    elif re.search(r'\.\d+$',c): role='duplicate_header'
    elif c=='IdleTime': role='requires_semantic_check'
    elif not numeric.notna().any(): role='empty_or_non_numeric'
    elif numeric.nunique()<2: role='constant_in_development'
    else: role='candidate'
    roles.append(dict(feature=c,role=role))
roles=pd.DataFrame(roles)
features=roles.loc[roles.role.eq('candidate'),'feature'].tolist()
for c in features:
    dev[c]=pd.to_numeric(dev[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
display(roles)
roles.to_csv(OUT/'feature_roles.csv',index=False)
print('Numeric candidates:',len(features))
redundant=[]
for a,b in combinations(features,2):
    if dev[a].equals(dev[b]): redundant.append({'feature_a':a,'feature_b':b})
redundant=pd.DataFrame(redundant,columns=['feature_a','feature_b'])
display(redundant)
redundant.to_csv(OUT/'identical_numeric_features.csv',index=False)
display(data.groupby(['traffic_label','congestion_level','SrcAddr','DstAddr','Dport'],dropna=False).size().rename('records').reset_index())


## 5. Distribution shifts within each traffic class
For each feature and pair of levels:
- KS distance (0–1): distribution separation, including non-monotonic changes.
- Signed Cliff's delta (−1–1): whether the second level tends to be larger; computed from Mann–Whitney U.
- Median and interquartile range: effect in original units; units remain those of the Argus export.

Use an equal-weight macro average over level pairs and classes, so larger files do not dominate.
No p-values are interpreted as independent-flow evidence: records are temporally dependent and each class/level has only one file.
Repeat on Start records to assess sensitivity to repeated Status records. Start-only is a diagnostic subset, not guaranteed complete sessions.


In [ ]:
def compare_levels(frame, positive_handshake=False):
    rows=[]
    for traffic in CLASSES:
        g=frame[frame.traffic_label.eq(traffic)]
        for f in features:
            for low,high in combinations(LEVELS,2):
                a0=g.loc[g.congestion_level.eq(low),f]
                b0=g.loc[g.congestion_level.eq(high),f]
                a,b=a0.dropna(),b0.dropna()
                if positive_handshake and f in ['TcpRtt','SynAck','AckDat']:
                    a,b=a[a>0],b[b>0]
                coverage_a=len(a)/max(1,len(a0)); coverage_b=len(b)/max(1,len(b0))
                if min(len(a),len(b))<MIN_VALID or min(coverage_a,coverage_b)<MIN_COVERAGE: continue
                u=mannwhitneyu(b,a,alternative='two-sided',method='asymptotic').statistic
                rows.append(dict(traffic_label=traffic,feature=f,level_a=low,level_b=high,
                    n_a=len(a),n_b=len(b),coverage_a=coverage_a,coverage_b=coverage_b,
                    ks=ks_2samp(a,b,method='asymp').statistic,
                    cliffs_delta=2*u/(len(a)*len(b))-1,
                    median_a=a.median(),median_b=b.median(),
                    iqr_a=a.quantile(.75)-a.quantile(.25),iqr_b=b.quantile(.75)-b.quantile(.25)))
    return pd.DataFrame(rows)
def rank_effects(effects):
    if effects.empty: return pd.DataFrame()
    per_class=effects.groupby(['feature','traffic_label']).agg(mean_ks=('ks','mean'),pair_count=('ks','size'))
    per_class=per_class[per_class.pair_count.eq(3)]
    ranked=per_class.groupby('feature').agg(macro_ks=('mean_ks','mean'),min_class_ks=('mean_ks','min'),
        max_class_ks=('mean_ks','max'),classes_available=('mean_ks','size'))
    return ranked.sort_values(['classes_available','macro_ks'],ascending=[False,False])
effects=compare_levels(dev)
ranking=rank_effects(effects)
start_effects=compare_levels(dev[dev.Cause.eq('Start')])
start_ranking=rank_effects(start_effects)
positive_effects=compare_levels(dev,positive_handshake=True)
comparison=ranking.join(start_ranking[['macro_ks']].rename(columns={'macro_ks':'start_only_ks'}),how='left')
display(comparison)
display(effects[effects.feature.isin(['TcpRtt','SynAck','AckDat'])])
for name,table in [('pairwise_effects',effects),('feature_ranking',comparison),('start_only_effects',start_effects),('positive_handshake_effects',positive_effects)]:
    table.to_csv(OUT/f'{name}.csv',index=name=='feature_ranking')


## 6. Temporal stability inside development data
Repeat ranking in the early and late halves of the development partition. Agreement is a stability diagnostic, not independent replication or causal attribution to congestion.


In [ ]:
early,late=[],[]
for _,g in dev.groupby('run_id',sort=False):
    cut=len(g)//2
    early.append(g.iloc[:cut]);late.append(g.iloc[cut:])
r_early=rank_effects(compare_levels(pd.concat(early)))
r_late=rank_effects(compare_levels(pd.concat(late)))
stability=comparison.join(r_early[['macro_ks']].rename(columns={'macro_ks':'early_ks'})).join(r_late[['macro_ks']].rename(columns={'macro_ks':'late_ks'}))
stability['early_late_gap']=(stability.early_ks-stability.late_ks).abs()
display(stability)
stability.to_csv(OUT/'ranking_stability.csv')


## 7. Plots: class-specific shifts and the paper's three timing features
Plots use development data only. A large effect in one class does not establish universal sensitivity.


In [ ]:
heat=effects.groupby(['feature','traffic_label']).ks.mean().unstack().reindex(index=ranking.index,columns=CLASSES)
fig,ax=plt.subplots(figsize=(9,max(5,len(heat)*.28)))
sns.heatmap(heat,vmin=0,vmax=1,cmap='viridis',ax=ax)
ax.set_title('Mean pairwise KS distance — development records')
fig.tight_layout();fig.savefig(OUT/'feature_shift_heatmap.png',dpi=160);plt.show()
fig,axes=plt.subplots(3,5,figsize=(18,10))
for i,f in enumerate(['TcpRtt','SynAck','AckDat']):
    for j,traffic in enumerate(CLASSES):
        ax=axes[i,j]
        subset=dev[dev.traffic_label.eq(traffic)]
        sns.boxplot(data=subset,x='congestion_level',y=f,order=LEVELS,showfliers=False,ax=ax)
        ax.set_yscale('symlog',linthresh=1e-5)
        ax.set_ylim(bottom=0)
        ax.set_title(f'{traffic}: {f}');ax.set_xlabel('')
fig.tight_layout();fig.savefig(OUT/'timing_features.png',dpi=160);plt.show()
# Spearman redundancy on development only, not used to select features automatically.
corr=dev[features].corr(method='spearman')
corr.to_csv(OUT/'spearman_correlations.csv')


## 8. Interpretation and next step
- Large KS means observed distribution shift, not proof that congestion is its sole cause. Capture date, VM scheduling, file mix, record timeouts and traffic direction can also differ.
- Source/destination statistics retain Argus orientation. Examine addresses/direction before interpreting forward versus reverse traffic.
- Missing inter-packet fields cannot be reconstructed from these CSVs. Re-export from PCAP with the required Argus measurements if those features are needed.
- `TcpRtt`, `SynAck`, `AckDat` may be algebraically dependent; do not present them as three independent mechanisms.
- Do not drop rows with zero timing silently. Compare the positive-only diagnostic and report coverage.
- The reserved holdout must remain outside feature selection. If these files have already informed previous choices, a fresh independent run is needed for a genuinely untouched final test.
- No sliding windows are constructed here. Later, create them within known sequences **after** partitioning; never cross file/split boundaries. Do not use the unknown class label to define inference-time sequences.
- Compare methods with the same selected features and splits; retain a full-feature comparison as a complementary experiment.

نتیجهٔ این مرحله یک فهرست پیشنهادی برای بررسی است، نه حذف خودکار ستون‌ها یا ادعای برتری مدل.


In [ ]:
manifest={'source_commit':SOURCE_COMMIT,'development_fraction':DEV_FRACTION,
          'min_valid':MIN_VALID,'min_coverage':MIN_COVERAGE,'seed':SEED,
          'files':inventory[['file','sha256']].to_dict('records'),
          'rows':len(data),'candidate_features':features,
          'notes':'Exploratory within-class ranking; no classifier trained; source files retained.'}
(OUT/'analysis_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Analysis complete. Reports:',OUT)
